# Week 03 — Data Cleaning & Preprocessing Pipeline
### Turn Raw Noisy Data into an ML-Ready Dataset
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

print("Preprocessing modules loaded!")


## 1. Load Raw Dataset & Inspect Data Issues


In [ ]:
df = pd.read_csv("datasets/customer_churn_raw.csv")
print(f"Initial Shape: {df.shape}")
df.head(10)


## 2. Duplicate Detection & Removal


In [ ]:
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")


## 3. Missing Value Analysis & Imputation


In [ ]:
print(df.isnull().sum())

# Median imputation for numerical features with skew/outliers
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Tenure_Months"] = df["Tenure_Months"].fillna(df["Tenure_Months"].median())
print("Missing values after imputation:
", df.isnull().sum())


## 4. Outlier Detection & Capping (IQR Method)


In [ ]:
Q1 = df["Monthly_Charges"].quantile(0.25)
Q3 = df["Monthly_Charges"].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Monthly Charges IQR: {IQR:.2f} | Upper Fence: {upper_fence:.2f}")

# Check outliers
outliers = df[df["Monthly_Charges"] > upper_fence]
print(f"Detected {len(outliers)} outlier(s):")
print(outliers[["CustomerID", "Monthly_Charges"]])

# Cap outliers to the upper fence
df["Monthly_Charges"] = np.where(df["Monthly_Charges"] > upper_fence, upper_fence, df["Monthly_Charges"])
print("Outliers capped successfully!")


## 5. Categorical Encoding


In [ ]:
# Binary encoding target variable
le = LabelEncoder()
df["Churn"] = le.fit_transform(df["Churn"])

# One-Hot Encoding nominal features
df_encoded = pd.get_dummies(df, columns=["Contract_Type", "Payment_Method"], drop_first=True, dtype=int)
df_encoded.head()


## 6. Train-Test Split & Feature Scaling


In [ ]:
X = df_encoded.drop(columns=["CustomerID", "Churn"])
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
num_features = ["Age", "Tenure_Months", "Monthly_Charges"]
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print(f"Train set: {X_train.shape} | Test set: {X_test.shape}")
print(X_train.head())


## 7. Export Cleaned Dataset & Summary Report


In [ ]:
df_encoded.to_csv("datasets/customer_churn_cleaned.csv", index=False)
print("Saved clean ML-ready dataset to datasets/customer_churn_cleaned.csv")
